# Démo de l'Orchestrateur

Ce notebook démontre l'utilisation de la classe `Orchestrator` pour coordonner le générateur, le chimiste et le biologiste.

In [1]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [3]:
import random
from typing import List, Dict, Any

from peptide_pipeline.orchestrator.orchestrator import Orchestrator
from peptide_pipeline.chemist.chemist_agent import ChemistAgent
from peptide_pipeline.biologist.esm_biologist_zscore import ESMBiologistZscore
from peptide_pipeline.generator.base import BaseGenerator

# Création d'un générateur factice pour la démo
class DemoGenerator(BaseGenerator):
    def generate_peptides(self, count: int) -> List[str]:
        aa = ['A','C','D','E','F','G','H','I','K','L','M','N','P','Q','R','S','T','V','W','Y']
        return ["".join(random.choices(aa, k=random.randint(5, 10))) for _ in range(count)]
    
    def modify_peptides(self, sequences: List[str], feedback: Dict[str, Any]) -> List[str]:
        return self.generate_peptides(feedback.get("count", len(sequences)))

ModuleNotFoundError: No module named 'torch'

In [ ]:
# 1. Initialisation des agents
generator = DemoGenerator()
chemist = ChemistAgent()
biologist = ESMBiologistZscore(
    reference_peptide="RLLKRFKHLFK",
    model_name="facebook/esm2_t12_35M_UR50D" # Petit modèle pour la démo
)

print("Agents initialisés avec succès.")

In [ ]:
# 2. Initialisation de l'orchestrateur
orchestrator = Orchestrator(
    generator=generator,
    chemist=chemist,
    biologist=biologist
)

print("Orchestrateur prêt.")

In [ ]:
# 3. Lancement de la pipeline
results = orchestrator.run(
    nb_iterations=3,
    nb_peptides=20,
    top_k=5,
    exploration_rate=0
)

print(f"\nRecherche terminée. {len(results)} candidats retournés.")

In [ ]:
# 4. Affichage des résultats
import pandas as pd

df_results = []
for res in results:
    row = {"Peptide": res["peptide"], "Score ESM": res["score"]}
    row.update(res["properties"])
    df_results.append(row)

df = pd.DataFrame(df_results)
df.head()